In [1]:
# Keep your existing helper functions as-is
def calculate_performance(results, test_emails):
    """
    Calculate performance metrics from prediction results.

    Args:
        results: List of PredictionResult objects
        test_emails: List of test email dictionaries with 'label' field

    Returns:
        dict: Performance metrics including correct, wrong, false_positives, false_negatives, accuracy
    """
    correct = 0
    wrong = 0
    false_positives = 0
    false_negatives = 0

    for i, result in enumerate(results):
        actual_label = test_emails[i]['label']
        predicted_label = 1 if result.predicted_label == 'PHISHING' else 0

        if predicted_label == actual_label:
            correct += 1
        else:
            wrong += 1
            if predicted_label == 1 and actual_label == 0:
                false_positives += 1
            elif predicted_label == 0 and actual_label == 1:
                false_negatives += 1

    total = len(results)
    accuracy = (correct / total) * 100

    return {
        'correct': correct,
        'wrong': wrong,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'total': total,
        'accuracy': accuracy
    }


def display_performance_summary(metrics, title="PERFORMANCE SUMMARY"):
    """
    Display performance summary in a formatted table.

    Args:
        metrics: Dictionary with performance metrics from calculate_performance()
        title: Title for the summary display
    """
    print("=" * 80)
    print(title)
    print("=" * 80)
    print(f"\nCorrect: {metrics['correct']}/{metrics['total']}")
    print(f"Wrong: {metrics['wrong']}/{metrics['total']}")
    print(f"Accuracy: {metrics['accuracy']:.1f}%")
    print(f"\nFalse Positives: {metrics['false_positives']} (legitimate emails flagged as phishing)")
    print(f"False Negatives: {metrics['false_negatives']} (phishing emails missed)")
    print()


def display_detailed_results(results, test_emails, title="DETAILED RESULTS"):
    """
    Display detailed prediction results in a formatted table.

    Args:
        results: List of PredictionResult objects
        test_emails: List of test email dictionaries
        title: Title for the results table
    """
    print("=" * 130)
    print(title)
    print("=" * 130)
    print(f"{'ID':<5} {'CATEGORY':<26} {'PREDICTED':<13} {'PHISH %':<11} {'RESULT':<16} {'SUBJECT':<50}")
    print("=" * 130)

    for i, result in enumerate(results):
        email = test_emails[i]
        actual_label = email['label']
        predicted_label = 1 if result.predicted_label == 'PHISHING' else 0

        # Determine if correct
        is_correct = predicted_label == actual_label

        # Determine result symbol and text
        if is_correct:
            result_text = "✅ CORRECT"
        else:
            if predicted_label == 1 and actual_label == 0:
                result_text = "❌ FALSE POS"
            else:
                result_text = "❌ FALSE NEG"

        # Format output
        phish_percent = f"{result.phishing_probability * 100:.1f}%"
        subject_short = email['subject'][:50]
        email_id = email.get('id', i + 1)

        print(f"{email_id:<5} {email['category']:<26} {result.predicted_label:<13} {phish_percent:<11} {result_text:<16} {subject_short}")

    print("=" * 130)


def compare_results(results_before, results_after, test_emails,
                   title_before="BEFORE", title_after="AFTER"):
    """
    Compare two sets of prediction results and display improvements.

    Args:
        results_before: List of PredictionResult objects from before
        results_after: List of PredictionResult objects from after
        test_emails: List of test email dictionaries
        title_before: Label for before results
        title_after: Label for after results
    """
    # Calculate metrics for both
    metrics_before = calculate_performance(results_before, test_emails)
    metrics_after = calculate_performance(results_after, test_emails)

    # Display comparison header
    print("\n" + "=" * 100)
    print(f"{title_before} vs {title_after} - PERFORMANCE COMPARISON".center(100))
    print("=" * 100)

    # Overall metrics comparison table
    print(f"\n{'METRIC':<25} | {title_before:^20} | {title_after:^20} | {'CHANGE':^20}")
    print("-" * 100)

    # Accuracy
    acc_change = metrics_after['accuracy'] - metrics_before['accuracy']
    acc_symbol = "📈" if acc_change > 0 else "📉" if acc_change < 0 else "➡️"
    print(f"{'Accuracy':<25} | {metrics_before['accuracy']:>19.1f}% | {metrics_after['accuracy']:>19.1f}% | "
          f"{acc_symbol} {acc_change:>+16.1f}%")

    # Correct predictions
    correct_change = metrics_after['correct'] - metrics_before['correct']
    correct_symbol = "✅" if correct_change > 0 else "❌" if correct_change < 0 else "➡️"
    print(f"{'Correct Predictions':<25} | {metrics_before['correct']:>15}/{metrics_before['total']:>2} | "
          f"{metrics_after['correct']:>15}/{metrics_after['total']:>2} | "
          f"{correct_symbol} {correct_change:>+16}")

    # False positives
    fp_change = metrics_after['false_positives'] - metrics_before['false_positives']
    fp_symbol = "✅" if fp_change < 0 else "❌" if fp_change > 0 else "➡️"
    print(f"{'False Positives':<25} | {metrics_before['false_positives']:>20} | "
          f"{metrics_after['false_positives']:>20} | "
          f"{fp_symbol} {fp_change:>+16}")

    # False negatives
    fn_change = metrics_after['false_negatives'] - metrics_before['false_negatives']
    fn_symbol = "✅" if fn_change < 0 else "❌" if fn_change > 0 else "➡️"
    print(f"{'False Negatives':<25} | {metrics_before['false_negatives']:>20} | "
          f"{metrics_after['false_negatives']:>20} | "
          f"{fn_symbol} {fn_change:>+16}")

    print("-" * 100)

    # Analyze changes in predictions
    corrected_emails = []
    regression_emails = []

    for i in range(len(results_before)):
        before_pred = 1 if results_before[i].predicted_label == 'PHISHING' else 0
        after_pred = 1 if results_after[i].predicted_label == 'PHISHING' else 0
        actual = test_emails[i]['label']

        # Check if prediction changed
        if before_pred != after_pred:
            email_info = {
                'id': test_emails[i].get('id', i + 1),
                'category': test_emails[i]['category'],
                'subject': test_emails[i]['subject'],
                'before_label': results_before[i].predicted_label,
                'after_label': results_after[i].predicted_label,
                'before_prob': results_before[i].phishing_probability * 100,
                'after_prob': results_after[i].phishing_probability * 100,
                'actual_label': 'PHISHING' if actual == 1 else 'LEGITIMATE'
            }

            # Corrected (wrong → right)
            if before_pred != actual and after_pred == actual:
                corrected_emails.append(email_info)

            # Regression (right → wrong)
            elif before_pred == actual and after_pred != actual:
                regression_emails.append(email_info)

    # Display corrected predictions
    if corrected_emails:
        print("\n" + "=" * 100)
        print(f"✅ CORRECTED PREDICTIONS ({len(corrected_emails)} emails)".center(100))
        print("=" * 100)

        for email in corrected_emails:
            print(f"\n📧 Email ID: {email['id']} | Category: {email['category']}")
            print(f"   Subject: {email['subject'][:80]}")
            print(f"   Actual Label: {email['actual_label']}")
            print(f"   ├─ BEFORE: {email['before_label']:<12} (Phishing: {email['before_prob']:>5.1f}%)")
            print(f"   └─ AFTER:  {email['after_label']:<12} (Phishing: {email['after_prob']:>5.1f}%) ✓")
    else:
        print("\n" + "=" * 100)
        print("NO CORRECTED PREDICTIONS".center(100))
        print("=" * 100)

    # Display regressions
    if regression_emails:
        print("\n" + "=" * 100)
        print(f"⚠️  REGRESSIONS ({len(regression_emails)} emails)".center(100))
        print("=" * 100)

        for email in regression_emails:
            print(f"\n📧 Email ID: {email['id']} | Category: {email['category']}")
            print(f"   Subject: {email['subject'][:80]}")
            print(f"   Actual Label: {email['actual_label']}")
            print(f"   ├─ BEFORE: {email['before_label']:<12} (Phishing: {email['before_prob']:>5.1f}%) ✓")
            print(f"   └─ AFTER:  {email['after_label']:<12} (Phishing: {email['after_prob']:>5.1f}%) ✗")

    # Summary
    print("\n" + "=" * 100)
    print("SUMMARY".center(100))
    print("=" * 100)
    print(f"  Total Corrections:  {len(corrected_emails)} emails")
    print(f"  Total Regressions:  {len(regression_emails)} emails")
    print(f"  Net Improvement:    {len(corrected_emails) - len(regression_emails):+} emails")

    if acc_change > 0:
        print(f"\n  🎉 Overall: Model performance IMPROVED by {acc_change:.1f}%")
    elif acc_change < 0:
        print(f"\n  ⚠️  Overall: Model performance DECREASED by {abs(acc_change):.1f}%")
    else:
        print(f"\n  ➡️  Overall: Model performance UNCHANGED")

    print("=" * 100)

    return {
        'before': metrics_before,
        'after': metrics_after,
        'corrected': len(corrected_emails),
        'regressions': len(regression_emails),
        'net_improvement': len(corrected_emails) - len(regression_emails)
    }

print("Helper functions loaded successfully")

Helper functions loaded successfully


In [2]:
from usage.phishing_detection.preprocessing.features import EmailFeatureExtractor
# Import new pipeline structure
from usage.phishing_detection.pipelines.prediction import PredictionPipeline
from usage.phishing_detection.preprocessing.cleaning import EmailCleaner

# Initialize components (reusable)
cleaner = EmailCleaner(verbose=False)

feature_extractor = EmailFeatureExtractor(
    subject_vectorizer_path='../models/pipeline_components/subject_vectorizer.pkl',
    body_vectorizer_path='../models/pipeline_components/body_vectorizer.pkl',
    verbose=False
)

# Create prediction pipeline
pipeline = PredictionPipeline(
    model_path='../models/v1_0/production/phishing_detector_mlp_classifier.pkl',
    cleaner=cleaner,
    feature_extractor=feature_extractor,
    threshold=75.0,  # Alert if phishing probability >= 75%
    verbose=False
)

print("Pipeline initialized successfully")

Pipeline initialized successfully


In [3]:
# Test dataset with realistic legitimate and phishing emails
test_emails = [
    # LEGITIMATE EMAILS (label=0)
    {
        'sender': 'GitHub <notifications@github.com>',
        'subject': 'Pull request merged in your repository',
        'body': '''Hi there,

Your pull request #342 has been successfully merged into the main branch.

Repository: CompanyName/project-api
Branch: feature/user-authentication → main

Changes included:
- Added JWT token validation
- Updated user schema
- Fixed authentication middleware bug

View the merged changes at:
https://github.com/CompanyName/project-api/pull/342

Best regards,
GitHub Team''',
        'label': 0,
        'category': 'Developer Notification'
    },
    {
        'sender': 'Jane Smith <jane.smith@mycompany.com>',
        'subject': 'Quarterly review meeting scheduled',
        'body': '''Hi team,

I've scheduled our Q1 review meeting for next Thursday, March 21st at 2:00 PM in Conference Room B.

Agenda:
- Review project milestones
- Discuss budget allocation
- Team performance updates

Please come prepared with your department reports.

Thanks,
Jane Smith
Senior Manager
Direct: (555) 123-4567''',
        'label': 0,
        'category': 'Work Communication'
    },
    {
        'sender': 'LinkedIn <messages-noreply@linkedin.com>',
        'subject': 'You have 3 new connection requests',
        'body': '''Hi,

You have new activity on LinkedIn:

3 people want to connect with you:
- Michael Johnson, Software Engineer at Tech Corp
- Sarah Williams, Product Manager at StartupXYZ
- David Brown, Data Scientist at AI Solutions

View all connection requests:
https://www.linkedin.com/mynetwork

Best regards,
The LinkedIn Team

LinkedIn Corporation
1000 W Maude Ave, Sunnyvale, CA 94085''',
        'label': 0,
        'category': 'Social Network'
    },
    {
        'sender': 'Sarah Johnson <sarah.johnson@university.edu>',
        'subject': 'Research paper collaboration opportunity',
        'body': '''Dear Colleague,

I hope this email finds you well. I came across your recent publication on machine learning applications in healthcare and found it fascinating.

I'm currently working on a similar project at University Medical Center and believe our research interests align well. Would you be interested in exploring a potential collaboration?

I'd be happy to schedule a call next week to discuss this further. Let me know your availability.

Best regards,
Sarah Johnson, PhD
Associate Professor, Department of Computer Science
University Medical Center
sarah.johnson@university.edu''',
        'label': 0,
        'category': 'Academic Communication'
    },
    {
        'sender': 'Netflix <info@account.netflix.com>',
        'subject': 'New episodes of your favorite shows are now available',
        'body': '''Hi there,

Good news! New episodes of shows you've been watching are now available:

- Stranger Things - Season 5, Episodes 1-3
- The Crown - Final Season Premiere
- Wednesday - Season 2, Episode 5

Continue watching where you left off, or discover something new in our recommendations for you.

Happy watching!
The Netflix Team

This email was sent to you because you are subscribed to Netflix updates.
Manage your email preferences in your account settings.''',
        'label': 0,
        'category': 'Entertainment Service'
    },
    {
        'sender': 'Dr. Michael Chen <michael.chen@healthcareclinic.com>',
        'subject': 'Appointment reminder - Annual checkup on Dec 22',
        'body': '''Dear Patient,

This is a reminder about your upcoming appointment:

Date: December 22, 2025
Time: 10:30 AM
Location: Healthcare Clinic, 456 Medical Plaza, Suite 200
Provider: Dr. Michael Chen

Please arrive 15 minutes early to complete any necessary paperwork. Remember to bring:
- Insurance card
- Photo ID
- List of current medications
- Any recent lab results

If you need to reschedule, please call us at (555) 234-5678 at least 24 hours in advance.

Best regards,
Healthcare Clinic Reception Team''',
        'label': 0,
        'category': 'Medical Appointment'
    },
    {
        'sender': 'Tom Wilson <tom.wilson@realestate.com>',
        'subject': 'Re: Property viewing follow-up',
        'body': '''Hi there,

Thank you for viewing the property at 789 Oak Street yesterday. I wanted to follow up and see if you have any additional questions.

As discussed, the property features:
- 3 bedrooms, 2.5 bathrooms
- Recently renovated kitchen
- Large backyard with mature trees
- Two-car garage
- Close to schools and shopping

The sellers are motivated and open to reasonable offers. If you're interested in making an offer or scheduling a second viewing, please let me know.

I'm also happy to provide you with comparable sales data in the neighborhood.

Best regards,
Tom Wilson
Licensed Real Estate Agent
Wilson Realty Group
(555) 345-6789''',
        'label': 0,
        'category': 'Real Estate'
    },
    {
        'sender': 'Marcus Brown <marcus.brown@techstartup.io>',
        'subject': 'Job application status update',
        'body': '''Dear Applicant,

Thank you for your interest in the Senior Software Engineer position at TechStartup Inc.

We have reviewed your application and are impressed with your background. We would like to invite you to the next stage of our interview process.

Next Steps:
1. Technical screening call (45 minutes) - Week of Jan 8-12
2. On-site interview (3 hours) - Tentatively Jan 15 or 16
3. Final decision by Jan 22

Please reply with your availability for the technical screening call. We're flexible with timing and happy to accommodate your schedule.

Looking forward to speaking with you!

Best regards,
Marcus Brown
Senior Recruiting Manager
TechStartup Inc.
marcus.brown@techstartup.io''',
        'label': 0,
        'category': 'Job Application'
    },
    {
        'sender': 'Emma Davis <emma.davis@freelance.com>',
        'subject': 'Project proposal - Website redesign',
        'body': '''Hi,

Thank you for reaching out about the website redesign project. I've reviewed your requirements and prepared a proposal.

Project Scope:
- Complete UI/UX redesign of main website
- Mobile-responsive design
- Implementation in React
- Integration with existing backend API
- 3 rounds of revisions included

Timeline: 8 weeks
Budget: $12,000

I've attached a detailed proposal document with mockups and project milestones. I'm confident I can deliver a modern, user-friendly website that meets your goals.

Would you like to schedule a call to discuss this further?

Best regards,
Emma Davis
Freelance Web Designer & Developer
Portfolio: www.emmadavis.design
emma.davis@freelance.com''',
        'label': 0,
        'category': 'Business Proposal'
    },
    {
        'sender': 'David Park <david.park@lawfirm.com>',
        'subject': 'Contract review completed',
        'body': '''Dear Client,

I have completed my review of the employment contract you forwarded last week. Overall, the terms are fairly standard, but I do have several recommendations:

Key Points:
1. Non-compete clause is overly broad - recommend negotiating geographic scope
2. IP assignment language should exclude pre-existing work
3. Severance terms are reasonable for this industry
4. Consider negotiating a sign-on bonus or equity compensation

I've marked up the contract with my suggested changes in track changes mode. Please review my comments and let me know if you'd like to discuss any points in detail.

I'm available for a call this week if needed.

Best regards,
David Park, Esq.
Employment Law Attorney
Park & Associates Law Firm
(555) 456-7890''',
        'label': 0,
        'category': 'Legal Services'
    },
    {
        'sender': 'Rachel Green <rachel.green@nonprofitorg.org>',
        'subject': 'Thank you for your generous donation',
        'body': '''Dear Supporter,

On behalf of everyone at Community Food Bank, I want to express our heartfelt gratitude for your generous donation of $500.

Your contribution will directly support our mission to provide nutritious meals to families in need. Last year, we served over 50,000 meals to children, seniors, and families facing food insecurity.

Impact of Your Gift:
- Provides meals for a family of four for two months
- Supports our weekend backpack program for children
- Helps maintain our community kitchen facilities

Your donation is tax-deductible. You will receive a formal acknowledgment letter for your records within 5-7 business days.

Thank you for making a difference in our community.

With gratitude,
Rachel Green
Development Director
Community Food Bank
rachel.green@nonprofitorg.org''',
        'label': 0,
        'category': 'Nonprofit Communication'
    },
    {
        'sender': 'James Miller <james.miller@gym.com>',
        'subject': 'Your fitness assessment results and personalized plan',
        'body': '''Hi there,

Great job completing your fitness assessment yesterday! I've analyzed your results and created a personalized 12-week training plan.

Current Fitness Level: Intermediate
Goals: Build strength, improve cardio endurance, lose 15 lbs

Your Customized Plan:
- Strength training: 3x per week (Mon/Wed/Fri)
- Cardio: 2x per week (Tue/Thu)
- Rest and recovery: Weekends
- Nutrition guidance included

I've attached your detailed workout plan and nutrition guide. Let's schedule your first training session for next Monday at 6:00 AM.

Remember, consistency is key! I'm here to support you every step of the way.

Let's crush these goals together!

James Miller
Certified Personal Trainer
FitLife Gym
james.miller@gym.com''',
        'label': 0,
        'category': 'Fitness Services'
    },

    # PHISHING EMAILS (label=1)
    {
        'sender': 'Apple Support <no-reply@apple-support-team.com>',
        'subject': 'Your Apple ID has been locked for security reasons',
        'body': '''Dear Apple Customer,

We detected unusual activity on your Apple ID account. For your security, we have temporarily locked your account.

IMMEDIATE ACTION REQUIRED:

Your account will be permanently disabled if you do not verify your identity within 24 hours.

To unlock your account, please verify your information:
http://apple-id-verification.com/secure/login?ref=8472938

Once verified, you will regain full access to:
- iCloud Storage
- App Store Purchases
- Apple Music
- iMessage and FaceTime

This is an automated security measure. If you do not verify within 24 hours, your account and all associated data will be permanently deleted.

Thank you for your immediate attention.

Apple Security Team
Cupertino, California''',
        'label': 1,
        'category': 'Account Security Scam'
    },
    {
        'sender': 'Microsoft Security <security-alert@microsoft-account.com>',
        'subject': 'Unusual sign-in activity detected on your account',
        'body': '''Microsoft Account Security Alert

We detected a sign-in attempt from an unrecognized device:

Location: Lagos, Nigeria
IP Address: 197.234.56.89
Device: Windows 10 PC
Date: December 19, 2025 at 3:42 AM

If this was not you, your account may be compromised.

SECURE YOUR ACCOUNT NOW:
Click here to review and secure your account: http://microsoft-security-check.com/verify

What's at risk:
- Your Outlook emails
- OneDrive files
- Office 365 documents
- Skype contacts

You must verify your identity within 12 hours or your account will be suspended for security reasons.

Microsoft Account Security Team
microsoft-account.com/security''',
        'label': 1,
        'category': 'Security Alert Scam'
    },
    {
        'sender': 'Amazon Customer Service <customer-service@amazon-support.net>',
        'subject': 'Your order #4829-5738-9283 requires payment verification',
        'body': '''Dear Valued Customer,

We were unable to process the payment for your recent order.

Order Details:
Order Number: #4829-5738-9283
Item: MacBook Pro 16" M3 Max
Amount: $3,499.00
Status: PAYMENT VERIFICATION REQUIRED

Your order will be cancelled if payment is not verified within 24 hours.

VERIFY PAYMENT NOW:
http://amazon-order-verification.com/payment?order=48295738

We apologize for any inconvenience. Once verification is complete, your order will ship immediately with expedited delivery.

If you did not place this order, please verify your account security immediately.

Customer Service Team
Amazon.com''',
        'label': 1,
        'category': 'Order Verification Scam'
    },
    {
        'sender': 'PayPal Security <service@paypal-secure.com>',
        'subject': 'Action Required: Verify your account to avoid limitations',
        'body': '''Important Notice from PayPal

Dear PayPal User,

We have detected unusual activity in your PayPal account. To protect your account, we have temporarily limited your access.

Recent suspicious activity:
- Login attempt from unknown location
- Large transaction pending review
- Account information mismatch

To restore full account access, please verify your information:
http://paypal-account-restore.com/verify

What's Limited:
✗ Sending money
✗ Withdrawing funds
✗ Receiving payments
✗ Adding bank accounts

This verification must be completed within 48 hours, or your account will be permanently suspended and any remaining balance will be forfeited.

We appreciate your immediate cooperation.

PayPal Account Review Department
www.paypal.com''',
        'label': 1,
        'category': 'Account Limitation Scam'
    },
    {
        'sender': 'Netflix Billing <billing@netflix-payment.com>',
        'subject': 'Your Netflix membership payment has failed',
        'body': '''Dear Netflix Member,

We were unable to process your monthly payment.

Account: john.doe@email.com
Plan: Premium (4 screens, Ultra HD)
Monthly charge: $19.99
Payment failed: December 19, 2025

Your membership will be cancelled in 24 hours if payment is not updated.

UPDATE PAYMENT METHOD:
http://netflix-billing-update.com/payment

What happens if you don't update:
- Your account will be cancelled
- You'll lose access to all content
- Your viewing history and preferences will be deleted
- You'll need to re-register as a new customer

Update your payment information now to continue enjoying Netflix.

The Netflix Team''',
        'label': 1,
        'category': 'Payment Update Scam'
    },
    {
        'sender': 'Wells Fargo Security <alerts@wellsfargo-security.com>',
        'subject': 'URGENT: Suspicious activity on your account',
        'body': '''Wells Fargo Fraud Alert

IMMEDIATE ACTION REQUIRED

We have detected the following suspicious transactions on your account:

Transaction 1: $2,847.00 - Best Buy (Los Angeles, CA)
Transaction 2: $1,593.00 - Apple Store (Miami, FL)
Transaction 3: $945.00 - Walmart (Houston, TX)

Total unauthorized charges: $5,385.00

These transactions have been flagged as potentially fraudulent.

If you did NOT authorize these charges, verify your account immediately:
http://wellsfargo-fraud-verification.com/secure

Your account will be frozen and all pending transactions will be processed if you do not respond within 6 hours.

Call us immediately: 1-800-FRAUD (This number cannot receive calls)

Wells Fargo Fraud Protection Team
Member FDIC''',
        'label': 1,
        'category': 'Bank Fraud Alert Scam'
    },
    {
        'sender': 'IRS Tax Notice <notices@irs-tax-department.com>',
        'subject': 'Final Notice: Tax Return Discrepancy - Action Required',
        'body': '''Internal Revenue Service
Official Tax Notice

Taxpayer ID: XXX-XX-1234
Tax Year: 2024
Notice Number: CP-8473-2938

URGENT TAX MATTER

We have identified a significant discrepancy in your 2024 tax return. Our records show you may be entitled to a tax refund of $4,847.00.

However, we have been unable to process your refund due to incomplete information.

CLAIM YOUR REFUND NOW:
http://irs-refund-processing.com/claim?id=8473

Information needed:
- Social Security Number verification
- Bank account details for direct deposit
- Identity confirmation

You must claim this refund within 72 hours or it will be forfeited to the U.S. Treasury.

This is your final notice. Failure to respond will result in:
- Loss of refund
- Potential audit
- Penalties and interest charges

Internal Revenue Service
U.S. Department of Treasury''',
        'label': 1,
        'category': 'Tax/IRS Scam'
    },
    {
        'sender': 'IT Department <it-support@yourcompany-secure.com>',
        'subject': 'Mandatory: Update your email password immediately',
        'body': '''URGENT NOTICE FROM IT DEPARTMENT

Dear Employee,

Due to a recent security breach, all employees must update their email passwords immediately.

SECURITY BREACH DETAILS:
- Date: December 18, 2025
- Systems affected: Email servers
- Compromised accounts: 847 employees

Your account is at risk of being compromised.

UPDATE YOUR PASSWORD NOW:
http://company-password-reset.com/secure

Steps to secure your account:
1. Click the link above
2. Enter your current username and password
3. Create a new strong password
4. Confirm your identity

DEADLINE: 6:00 PM TODAY

Accounts that are not updated by the deadline will be automatically suspended and all emails will be deleted.

If you have any questions, DO NOT reply to this email. Visit our IT Help Desk.

IT Security Team
Your Company Name''',
        'label': 1,
        'category': 'Corporate IT Scam'
    }
]

print(f"Test dataset created: {len(test_emails)} emails")
print(f"  Legitimate: {sum(1 for e in test_emails if e['label'] == 0)}")
print(f"  Phishing: {sum(1 for e in test_emails if e['label'] == 1)}")

Test dataset created: 20 emails
  Legitimate: 12
  Phishing: 8


In [4]:
# Run batch prediction
results = pipeline.predict_batch(
    emails=[{'sender': e['sender'], 'subject': e['subject'], 'body': e['body']}
            for e in test_emails],
    verbose=False
)

print("Batch prediction completed")

[PredictionPipeline] Starting batch prediction for 20 emails...
[PredictionPipeline] Starting email prediction...
[EmailCleaner] Starting email cleaning pipeline...
[EmailCleaner] Email cleaning pipeline completed successfully.

[EmailFeatureExtractor] Starting feature extraction pipeline...
[EmailFeatureExtractor] Feature extraction pipeline completed successfully.

[PredictionPipeline] Prediction complete: LEGITIMATE (confidence: 91.2%)

[PredictionPipeline] Starting email prediction...
[EmailCleaner] Starting email cleaning pipeline...
[EmailCleaner] Email cleaning pipeline completed successfully.

[EmailFeatureExtractor] Starting feature extraction pipeline...
[EmailFeatureExtractor] Feature extraction pipeline completed successfully.

[PredictionPipeline] Prediction complete: LEGITIMATE (confidence: 99.2%)

[PredictionPipeline] Starting email prediction...
[EmailCleaner] Starting email cleaning pipeline...
[EmailCleaner] Email cleaning pipeline completed successfully.

[EmailFeatu

In [5]:
# Calculate and display performance metrics
metrics = calculate_performance(results, test_emails)
display_performance_summary(metrics, title="BATCH PREDICTION PERFORMANCE")

BATCH PREDICTION PERFORMANCE

Correct: 18/20
Wrong: 2/20
Accuracy: 90.0%

False Positives: 2 (legitimate emails flagged as phishing)
False Negatives: 0 (phishing emails missed)



In [6]:
# Display detailed results
display_detailed_results(results, test_emails, title="DETAILED PREDICTION RESULTS")

DETAILED PREDICTION RESULTS
ID    CATEGORY                   PREDICTED     PHISH %     RESULT           SUBJECT                                           
1     Developer Notification     LEGITIMATE    4.4%        ✅ CORRECT        Pull request merged in your repository
2     Work Communication         LEGITIMATE    0.4%        ✅ CORRECT        Quarterly review meeting scheduled
3     Social Network             LEGITIMATE    21.4%       ✅ CORRECT        You have 3 new connection requests
4     Academic Communication     LEGITIMATE    0.2%        ✅ CORRECT        Research paper collaboration opportunity
5     Entertainment Service      LEGITIMATE    32.0%       ✅ CORRECT        New episodes of your favorite shows are now availa
6     Medical Appointment        LEGITIMATE    25.4%       ✅ CORRECT        Appointment reminder - Annual checkup on Dec 22
7     Real Estate                LEGITIMATE    3.5%        ✅ CORRECT        Re: Property viewing follow-up
8     Job Application            

## Online Learning

In [8]:
# Dataset for demonstrating online learning improvement
# SAME DATASET AS web3.py - ensures apples-to-apples comparison
test_emails_online = [
    # =====================================================================
    # E-COMMERCE ORDER CONFIRMATIONS (5)
    # =====================================================================
    {
        'id': 1,
        'category': 'Order Confirmation',
        'sender': 'Amazon <auto-confirm@amazon.com>',
        'subject': 'Your Amazon.com order #112-8472956-3847291',
        'body': '''Hello,

Thank you for your order. Your order has been received and is being processed.

Order Details:
Order #112-8472956-3847291
Order Date: December 13, 2025

Items Ordered:
- Wireless Headphones (Black) - Qty: 1 - $79.99
- USB-C Cable (6ft) - Qty: 2 - $12.99

Subtotal: $105.97
Shipping: FREE
Tax: $9.01
Order Total: $114.98

Estimated Delivery: December 16-18, 2025

Track your package: https://www.amazon.com/progress-tracker/package/

Shipping Address:
John Smith
123 Main Street
New York, NY 10001

Thank you for shopping with Amazon.

Amazon.com''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 2,
        'category': 'Order Confirmation',
        'sender': 'Best Buy <BestBuyInfo@emailinfo.bestbuy.com>',
        'subject': 'Your Best Buy order BBY01-482956374829 is confirmed',
        'body': '''Thank you for your order!

Order Number: BBY01-482956374829
Order Date: December 13, 2025

Order Summary:
- Samsung 55" 4K Smart TV - $599.99
- HDMI Cable - $19.99

Subtotal: $619.98
Tax: $52.70
Total: $672.68

Store Pickup:
Best Buy - 456 Market St, San Francisco, CA
Ready for pickup: December 14, 2025

We'll send you an email when your order is ready.

View order status: https://www.bestbuy.com/orders/

Questions? Visit BestBuy.com/support

Best Buy''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 3,
        'category': 'Shipping Notification',
        'sender': 'Target <guest.orders@target.com>',
        'subject': 'Your Target order is on the way - Arrives Dec 15',
        'body': '''Your order has shipped!

Order #827-4820-9283
Shipped on: December 13, 2025
Estimated delivery: December 15, 2025

What's in this shipment:
- Coffee Maker (Black) - $89.99
- Coffee Filters (100ct) - $4.99

Tracking Number: 1Z9999W99999999999
Carrier: UPS

Track your package: https://www.target.com/orders/track

Delivery address:
Sarah Johnson
789 Oak Avenue
Chicago, IL 60601

Thanks for shopping at Target!

Target''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 4,
        'category': 'Order Confirmation',
        'sender': 'Apple Store <no_reply@email.apple.com>',
        'subject': 'Your Apple Store order M482957392 is confirmed',
        'body': '''Thank you for your order.

Order Number: M482957392
Order Date: December 13, 2025

Items:
AirPods Pro (2nd generation) - $249.00
AppleCare+ for AirPods Pro - $29.00

Subtotal: $278.00
Estimated Tax: $23.63
Total: $301.63

Delivery Estimate: December 16-18, 2025

Track your order: https://www.apple.com/shop/order/list

Shipping to:
Michael Chen
321 Pine Street
Seattle, WA 98101

Thank you for your purchase.

Apple Store''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 5,
        'category': 'Delivery Confirmation',
        'sender': 'Walmart <WalmartOnline@walmart.com>',
        'subject': 'Your Walmart order was delivered',
        'body': '''Your order has been delivered.

Order #948-2847-3829
Delivered on: December 13, 2025 at 2:34 PM

Items delivered:
- Laundry Detergent (100 oz) - $14.99
- Paper Towels (12 rolls) - $19.99
- Dish Soap (3 pack) - $8.99

Order total: $43.97

Left at: Front door

View receipt: https://www.walmart.com/orders/

Rate your delivery experience.

Thank you for shopping with Walmart!

Walmart''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },

    # =====================================================================
    # PAYMENT RECEIPTS (5)
    # =====================================================================
    {
        'id': 6,
        'category': 'Payment Receipt',
        'sender': 'PayPal <service@paypal.com>',
        'subject': 'Receipt for your payment to Adobe Inc.',
        'body': '''You sent a payment

Transaction ID: 8JH29384KD920384
Date: December 13, 2025

To: Adobe Inc.
Amount: $54.99 USD

Payment for: Adobe Creative Cloud - Monthly Subscription

Payment method: Bank account ending in 4829

View transaction details: https://www.paypal.com/activity/payment/8JH29384KD920384

Questions? Visit the Resolution Center.

PayPal''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 7,
        'category': 'Payment Receipt',
        'sender': 'Stripe <receipts@stripe.com>',
        'subject': 'Receipt from Notion Labs [Receipt #8392-4820]',
        'body': '''Receipt from Notion Labs

Amount paid: $10.00
Date paid: December 13, 2025

Receipt Number: 8392-4820

Payment method: Visa ending in 4242

Description: Notion Pro - Monthly subscription

View receipt: https://invoice.stripe.com/i/acct_abc123/inv_xyz789

If you have questions, contact Notion Labs.

Stripe''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 8,
        'category': 'Payment Confirmation',
        'sender': 'Square <receipts@messaging.squareup.com>',
        'subject': 'Receipt for $47.50 at Coffee House',
        'body': '''Thanks for your purchase!

Coffee House
123 Main St, Portland, OR

Date: December 13, 2025
Time: 8:45 AM

Items:
2x Latte - $5.00 each
1x Croissant - $4.50
1x Avocado Toast - $12.00
Subtotal: $26.50

Tax: $2.25
Tip: $18.75
Total: $47.50

Paid with Visa ending in 5678

View full receipt: https://square.com/receipt/xyz789

Square, Inc.''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 9,
        'category': 'Payment Receipt',
        'sender': 'Venmo <venmo@venmo.com>',
        'subject': 'You paid Sarah Johnson $45.00',
        'body': '''You paid Sarah Johnson

Amount: $45.00
Date: December 13, 2025
Note: Dinner split

Paid from: Bank account ending in 8492

Transaction ID: 3847293847

View payment: https://venmo.com/transaction/3847293847

Your Venmo balance: $127.50

Venmo''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 10,
        'category': 'Invoice Payment',
        'sender': 'QuickBooks <notifications@intuit.com>',
        'subject': 'Payment received: Invoice #INV-1248 - $1,250.00',
        'body': '''Payment Received

Invoice: INV-1248
Amount: $1,250.00
Date: December 13, 2025

From: Tech Solutions LLC
For: Web Development Services - November 2025

Payment method: ACH Bank Transfer

Invoice now marked as PAID.

View invoice: https://quickbooks.intuit.com/invoice/INV-1248

QuickBooks Online''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },

    # =====================================================================
    # BOOKING/RESERVATION CONFIRMATIONS (5)
    # =====================================================================
    {
        'id': 11,
        'category': 'Flight Confirmation',
        'sender': 'Delta Air Lines <delta@emails.delta.com>',
        'subject': 'eTicket receipt - Confirmation #KL8D9F',
        'body': '''Your trip is confirmed

Confirmation Code: KL8D9F
Ticket Number: 0062847293847

Flight Details:
Delta Flight 1234
December 20, 2025

Depart: New York (JFK) 8:00 AM
Arrive: Los Angeles (LAX) 11:30 AM

Passenger: John Smith
Seat: 12A

Total Paid: $387.60

View reservation: https://www.delta.com/mytrips/

Check in starting 24 hours before departure.

Delta Air Lines''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 12,
        'category': 'Hotel Confirmation',
        'sender': 'Marriott Hotels <reservations@marriott.com>',
        'subject': 'Your reservation at Marriott Downtown - Conf# 847293847',
        'body': '''Reservation Confirmed

Confirmation Number: 847293847

Marriott Downtown San Francisco
123 Market Street, San Francisco, CA 94103

Check-in: December 18, 2025 (3:00 PM)
Check-out: December 21, 2025 (12:00 PM)
Nights: 3

Room: King Bed, City View
Guests: 2 Adults

Rate: $189.00 per night
Total: $567.00 (before taxes)

Cancellation: Free cancellation until December 17

Manage reservation: https://www.marriott.com/reservation/

Marriott Bonvoy Member: Yes

Marriott International''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 13,
        'category': 'Car Rental Confirmation',
        'sender': 'Enterprise Rent-A-Car <donotreply@enterprise.com>',
        'subject': 'Reservation confirmed - Confirmation #J8472938',
        'body': '''Your rental is confirmed

Confirmation Number: J8472938

Pick-up:
Location: San Francisco Airport (SFO)
Date: December 18, 2025 at 12:00 PM

Drop-off:
Location: San Francisco Airport (SFO)
Date: December 21, 2025 at 12:00 PM

Vehicle: Toyota Camry or similar (Intermediate)
Rate: $45.00 per day
Estimated Total: $135.00 (excluding taxes)

Driver: Michael Chen
Age: 28

Modify reservation: https://www.enterprise.com/reserve/

Bring your driver's license and credit card.

Enterprise Rent-A-Car''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 14,
        'category': 'Restaurant Reservation',
        'sender': 'OpenTable <dining@opentable.com>',
        'subject': 'Reservation confirmed at The Steakhouse - Dec 15',
        'body': '''Your reservation is confirmed

The Steakhouse
456 Fine Dining Ave, New York, NY

Date: December 15, 2025
Time: 7:30 PM
Party size: 4 people

Confirmation: RT8473928

Special requests: Window table preferred

Manage reservation: https://www.opentable.com/booking/

Reservation policies:
- Please arrive on time
- Cancellation: Up to 2 hours before

Add to calendar

OpenTable''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 15,
        'category': 'Event Ticket',
        'sender': 'Ticketmaster <noreply@ticketmaster.com>',
        'subject': 'Your tickets for Concert at Madison Square Garden',
        'body': '''Your order is confirmed

Order Number: 18-4729/NY
Event: Rock Band World Tour 2025
Venue: Madison Square Garden, New York, NY
Date: December 28, 2025 at 8:00 PM

Tickets: 2x Section 104, Row F, Seats 12-13
Price: $125.00 each
Total: $250.00

Your tickets are in your account.

View tickets: https://www.ticketmaster.com/myaccount/

Download the Ticketmaster app for mobile entry.

Ticketmaster''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },

    # =====================================================================
    # SUBSCRIPTION/SERVICE RECEIPTS (5)
    # =====================================================================
    {
        'id': 16,
        'category': 'Subscription Receipt',
        'sender': 'Spotify <no-reply@spotify.com>',
        'subject': 'Your receipt from Spotify',
        'body': '''Thanks for being a Premium member!

Receipt for: Spotify Premium
Period: December 2025

Amount: $9.99
Payment date: December 13, 2025
Payment method: Visa ending in 4829

Next billing date: January 13, 2026

View receipt: https://www.spotify.com/account/subscription/

Manage subscription: https://www.spotify.com/account/

Questions? Visit our support page.

Spotify''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 17,
        'category': 'Subscription Renewal',
        'sender': 'Microsoft 365 <microsoft365@mail.microsoft.com>',
        'subject': 'Your Microsoft 365 subscription has renewed',
        'body': '''Your subscription has been renewed

Microsoft 365 Family
Subscription renewed: December 13, 2025

Amount charged: $99.99
Payment method: Credit card ending in 3847
Next renewal: December 13, 2026

Subscription includes:
- Office apps for up to 6 people
- 1 TB OneDrive storage per person
- Advanced security

Manage subscription: https://account.microsoft.com/services/

Invoice: https://account.microsoft.com/billing/

Microsoft Corporation''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 18,
        'category': 'Subscription Payment',
        'sender': 'Adobe <message@adobe.com>',
        'subject': 'Adobe Creative Cloud payment confirmation',
        'body': '''Payment confirmed

Adobe Creative Cloud - All Apps Plan

Payment date: December 13, 2025
Amount: $54.99
Payment method: PayPal

Plan details:
- All Adobe apps
- 100 GB cloud storage
- Adobe Fonts
- Adobe Portfolio

Next billing: January 13, 2026

Manage plan: https://account.adobe.com/plans

View invoice: https://account.adobe.com/billing/

Adobe Systems''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 19,
        'category': 'Utility Bill',
        'sender': 'PG&E <customerservice@pge.com>',
        'subject': 'Your PG&E bill is ready - Amount due: $127.45',
        'body': '''Your bill is ready

Account: 8472-9384-7293
Bill date: December 13, 2025
Amount due: $127.45
Due date: January 2, 2026

Current charges:
Electric: $89.30
Gas: $38.15

Previous balance: $0.00

View bill: https://www.pge.com/myaccount/

Payment options:
- Pay online
- Auto pay
- Phone: 1-800-743-5000

Pacific Gas and Electric Company''',
        'label': 0,
        'expected': 'LEGITIMATE'
    },
    {
        'id': 20,
        'category': 'Domain Renewal',
        'sender': 'GoDaddy <notify@godaddy.com>',
        'subject': 'Domain renewal receipt - mywebsite.com',
        'body': '''Domain renewal confirmed

Domain: mywebsite.com
Renewed: December 13, 2025
Expires: December 13, 2026

Amount: $17.99
Payment method: Credit card ending in 4829

Order #: 483729384

Auto-renewal: Enabled

Manage domain: https://dcc.godaddy.com/domains/

View invoice: https://account.godaddy.com/orders/

GoDaddy''',
        'label': 0,
        'expected': 'LEGITIMATE'
    }
]

print(f"Online learning test dataset created: {len(test_emails_online)} emails")
print(f"  All are LEGITIMATE e-commerce/payment emails (label=0)")
print(f"  This is the SAME dataset used in web3.py for comparison")

Online learning test dataset created: 20 emails
  All are LEGITIMATE e-commerce/payment emails (label=0)
  This is the SAME dataset used in web3.py for comparison


In [9]:
# Prepare Playground Test Data
# Convert test_emails_online to playground format and export

import json
import os
from datetime import datetime

print(f"Loaded {len(test_emails_online)} emails")

# Convert to playground format
playground_data = [
    {
        'sender': email['sender'],
        'subject': email['subject'],
        'body': email['body'],
        'label': email['label']
    }
    for email in test_emails_online
]

print(f"Converted {len(playground_data)} emails")

# Export to JSON
output_dir = 'storage/playground_data'
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'playground_test_data_{timestamp}.json'
filepath = os.path.join(output_dir, filename)

with open(filepath, 'w', encoding='utf-8') as f:
    json.dump(playground_data, f, indent=2, ensure_ascii=False)

print(f"Exported to: {filepath}")
print("Ready for playground import")

Loaded 20 emails
Converted 20 emails
Exported to: storage/playground_data\playground_test_data_20251221_140704.json
Ready for playground import


In [10]:
# Run batch prediction BEFORE online learning
print("=" * 80)
print("PREDICTING E-COMMERCE EMAILS - BEFORE ONLINE LEARNING")
print("=" * 80)
print()

results_before = pipeline.predict_batch(
    emails=[{'sender': e['sender'], 'subject': e['subject'], 'body': e['body']}
            for e in test_emails_online],
    verbose=False
)

print("Prediction completed")

PREDICTING E-COMMERCE EMAILS - BEFORE ONLINE LEARNING

[PredictionPipeline] Starting batch prediction for 20 emails...
[PredictionPipeline] Starting email prediction...
[EmailCleaner] Starting email cleaning pipeline...
[EmailCleaner] Email cleaning pipeline completed successfully.

[EmailFeatureExtractor] Starting feature extraction pipeline...
[EmailFeatureExtractor] Feature extraction pipeline completed successfully.

[PredictionPipeline] Prediction complete: LEGITIMATE (confidence: 92.2%)

[PredictionPipeline] Starting email prediction...
[EmailCleaner] Starting email cleaning pipeline...
[EmailCleaner] Email cleaning pipeline completed successfully.

[EmailFeatureExtractor] Starting feature extraction pipeline...
[EmailFeatureExtractor] Feature extraction pipeline completed successfully.

[PredictionPipeline] Prediction complete: PHISHING (confidence: 59.3%)

[PredictionPipeline] Starting email prediction...
[EmailCleaner] Starting email cleaning pipeline...
[EmailCleaner] Email c

In [11]:
# Calculate and display performance BEFORE online learning
metrics_before = calculate_performance(results_before, test_emails_online)
display_performance_summary(metrics_before, title="PERFORMANCE BEFORE ONLINE LEARNING")
print("Note: Model is too aggressive on e-commerce/payment emails")

PERFORMANCE BEFORE ONLINE LEARNING

Correct: 12/20
Wrong: 8/20
Accuracy: 60.0%

False Positives: 8 (legitimate emails flagged as phishing)
False Negatives: 0 (phishing emails missed)

Note: Model is too aggressive on e-commerce/payment emails


In [12]:
# Display detailed results BEFORE online learning
display_detailed_results(results_before, test_emails_online,
                        title="PREDICTIONS BEFORE ONLINE LEARNING")

PREDICTIONS BEFORE ONLINE LEARNING
ID    CATEGORY                   PREDICTED     PHISH %     RESULT           SUBJECT                                           
1     Order Confirmation         LEGITIMATE    3.9%        ✅ CORRECT        Your Amazon.com order #112-8472956-3847291
2     Order Confirmation         PHISHING      79.7%       ❌ FALSE POS      Your Best Buy order BBY01-482956374829 is confirme
3     Shipping Notification      PHISHING      89.3%       ❌ FALSE POS      Your Target order is on the way - Arrives Dec 15
4     Order Confirmation         LEGITIMATE    18.2%       ✅ CORRECT        Your Apple Store order M482957392 is confirmed
5     Delivery Confirmation      PHISHING      80.6%       ❌ FALSE POS      Your Walmart order was delivered
6     Payment Receipt            PHISHING      91.7%       ❌ FALSE POS      Receipt for your payment to Adobe Inc.
7     Payment Receipt            LEGITIMATE    13.1%       ✅ CORRECT        Receipt from Notion Labs [Receipt #8392-4820

In [13]:
from usage.dataset.legitimate import legitimate_emails, legitimate_shipping_notifications, \
    legitimate_subscription_renewals
from usage.dataset.phishing import phishing_emails, phishing_shipping_notifications, phishing_subscription_renewals

# Prepare training data with user corrections
# new_emails = legitimate_emails + phishing_emails
new_emails = (
    phishing_shipping_notifications +      # 10 phishing shipping
    legitimate_shipping_notifications +     # 10 legitimate shipping
    phishing_subscription_renewals +        # 10 phishing subscriptions
    legitimate_subscription_renewals +       # 10 legitimate subscriptions
    legitimate_emails + phishing_emails # original training emails
)

print(f"Training data prepared: {len(new_emails)} emails")
print(f"  Legitimate: {len(legitimate_emails)}")
print(f"  Phishing: {len(phishing_emails)}")

Created 10 phishing shipping notification emails
✓ Red flags included: fake tracking links, payment requests, urgent timeframes, address verification scams
Created 10 phishing subscription renewal emails
✓ Red flags included: fake renewal notices, cancel/refund urgency, payment verification requests, suspicious domains
Created 10 legitimate shipping notification emails
✓ Legitimate patterns: Official domains (@ups.com, @amazon.com), no payment requests, no urgency, helpful tracking info
Created 10 legitimate subscription renewal emails
✓ Legitimate patterns: Official domains, clear pricing, no urgency, manage subscription links, professional tone
Training data prepared: 127 emails
  Legitimate: 46
  Phishing: 41


In [14]:
# Import online learning pipeline
from usage.phishing_detection.pipelines.training import OnlineLearningPipeline

# Initialize online learning pipeline (uses same cleaner and feature_extractor)
training_pipeline = OnlineLearningPipeline(
    base_model_path='../models/v1_0/production/phishing_detector_mlp_classifier.pkl',
    cleaner=cleaner,  # Reuse from earlier
    feature_extractor=feature_extractor,  # Reuse from earlier
    output_dir='../models',
    verbose=True
)

print("\nStarting online learning with user corrections...")
print("=" * 80)

# Perform online learning
result = training_pipeline.partial_fit_batch(
    emails=new_emails,
    parent_version='v1_0',
    validate=True
)

print("=" * 80)

if result.success:
    print(f"\n✅ Training successful!")
    print(f"   New version created: {result.version_number}")
    print(f"   Emails processed: {result.emails_processed}")
    print(f"   Accuracy before: {result.performance_before['accuracy']:.3f}")
    print(f"   Accuracy after: {result.performance_after['accuracy']:.3f}")
else:
    print(f"\n❌ Training failed: {result.error}")

[OnlineLearningPipeline] Initialized
[OnlineLearningPipeline] Base model: ../models/v1_0/production/phishing_detector_mlp_classifier.pkl
[OnlineLearningPipeline] Output directory: ../models

Starting online learning with user corrections...

[OnlineLearningPipeline] ===== ONLINE LEARNING PIPELINE =====
Batch size: 127
[OnlineLearningPipeline] [0%] Calculating version number
[OnlineLearningPipeline] New version: v1_7
[OnlineLearningPipeline] [10%] Calculating version number completed
[OnlineLearningPipeline] [10%] Loading base model
[OnlineLearningPipeline] Model loaded: MLPClassifier
[OnlineLearningPipeline] [20%] Loading base model completed
[OnlineLearningPipeline] [20%] Preprocessing batch
[OnlineLearningPipeline] Preprocessing 127 emails
[EmailCleaner] Starting email cleaning pipeline...
[EmailCleaner] Email cleaning pipeline completed successfully.

[EmailFeatureExtractor] Starting feature extraction pipeline...
[EmailFeatureExtractor] Feature extraction pipeline completed success

In [15]:
# Create new pipeline with updated model
pipeline_v2 = PredictionPipeline(
    model_path=f'../models/{result.version_number}/production/phishing_detector_mlp_classifier.pkl',
    cleaner=cleaner,  # Reuse same cleaner
    feature_extractor=feature_extractor,  # Reuse same feature_extractor
    threshold=75.0,
    verbose=False
)

print(f"New pipeline initialized with model version: {result.version_number}")

New pipeline initialized with model version: v1_7


In [16]:
# Run batch prediction AFTER online learning
print("\n" + "=" * 80)
print("PREDICTING E-COMMERCE EMAILS - AFTER ONLINE LEARNING")
print("=" * 80)
print()

results_after = pipeline_v2.predict_batch(
    emails=[{'sender': e['sender'], 'subject': e['subject'], 'body': e['body']}
            for e in test_emails_online],
    verbose=False
)

print("Prediction completed")


PREDICTING E-COMMERCE EMAILS - AFTER ONLINE LEARNING

[PredictionPipeline] Starting batch prediction for 20 emails...
[PredictionPipeline] Starting email prediction...
[EmailCleaner] Starting email cleaning pipeline...
[EmailCleaner] Email cleaning pipeline completed successfully.

[EmailFeatureExtractor] Starting feature extraction pipeline...
[EmailFeatureExtractor] Feature extraction pipeline completed successfully.

[PredictionPipeline] Prediction complete: LEGITIMATE (confidence: 99.0%)

[PredictionPipeline] Starting email prediction...
[EmailCleaner] Starting email cleaning pipeline...
[EmailCleaner] Email cleaning pipeline completed successfully.

[EmailFeatureExtractor] Starting feature extraction pipeline...
[EmailFeatureExtractor] Feature extraction pipeline completed successfully.

[PredictionPipeline] Prediction complete: LEGITIMATE (confidence: 82.4%)

[PredictionPipeline] Starting email prediction...
[EmailCleaner] Starting email cleaning pipeline...
[EmailCleaner] Email

In [17]:
# Calculate and display performance AFTER online learning
metrics_after = calculate_performance(results_after, test_emails_online)
display_performance_summary(metrics_after, title="PERFORMANCE AFTER ONLINE LEARNING")

PERFORMANCE AFTER ONLINE LEARNING

Correct: 18/20
Wrong: 2/20
Accuracy: 90.0%

False Positives: 2 (legitimate emails flagged as phishing)
False Negatives: 0 (phishing emails missed)



In [18]:
# Display detailed results AFTER online learning
display_detailed_results(results_after, test_emails_online,
                        title="PREDICTIONS AFTER ONLINE LEARNING")

PREDICTIONS AFTER ONLINE LEARNING
ID    CATEGORY                   PREDICTED     PHISH %     RESULT           SUBJECT                                           
1     Order Confirmation         LEGITIMATE    0.5%        ✅ CORRECT        Your Amazon.com order #112-8472956-3847291
2     Order Confirmation         LEGITIMATE    8.8%        ✅ CORRECT        Your Best Buy order BBY01-482956374829 is confirme
3     Shipping Notification      LEGITIMATE    28.4%       ✅ CORRECT        Your Target order is on the way - Arrives Dec 15
4     Order Confirmation         LEGITIMATE    1.4%        ✅ CORRECT        Your Apple Store order M482957392 is confirmed
5     Delivery Confirmation      LEGITIMATE    15.7%       ✅ CORRECT        Your Walmart order was delivered
6     Payment Receipt            LEGITIMATE    42.8%       ✅ CORRECT        Receipt for your payment to Adobe Inc.
7     Payment Receipt            LEGITIMATE    1.6%        ✅ CORRECT        Receipt from Notion Labs [Receipt #8392-4820]

In [45]:
# Compare results and show improvements
comparison = compare_results(results_before, results_after, test_emails_online,
                            title_before="BEFORE LEARNING", title_after="AFTER LEARNING")


                     BEFORE LEARNING vs AFTER LEARNING - PERFORMANCE COMPARISON                     

METRIC                    |   BEFORE LEARNING    |    AFTER LEARNING    |        CHANGE       
----------------------------------------------------------------------------------------------------
Accuracy                  |                60.0% |                90.0% | 📈            +30.0%
Correct Predictions       |              12/20 |              18/20 | ✅               +6
False Positives           |                    8 |                    2 | ✅               -6
False Negatives           |                    0 |                    0 | ➡️               +0
----------------------------------------------------------------------------------------------------

                                 ✅ CORRECTED PREDICTIONS (6 emails)                                 

📧 Email ID: 2 | Category: Order Confirmation
   Subject: Your Best Buy order BBY01-482956374829 is confirmed
   Actual Label: L